# MMAI 5040 — Session 6 | StreamWave Recommender Lab

In Session 5, K-Means identified **four customer segments** using StreamWave subscriber data. For C0 (sports-focused accounts), we proposed opt-in sports event reminders. 

### Files you need

1. `Session 5_streamwave_subscribers.csv` — the **same original subscriber dataset** used in Session 5 (1,590 accounts; 13 clustering features).
2. `Session 6_streamwave_sports_events.csv` — a **new synthetic interaction dataset** used only for this recommendation exercise.



## Part A — Reuse the StreamWave clustering results

We begin with the **same 13 subscriber features** and `k = 4` from Session 5 (`random_state=42`, `n_init=25`). This lab reproduces the cluster assignments so that you can run it independently of the Session 5 notebook.


In [4]:

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# Specify the full path for each CSV file
path1 = r"Session 5_streamwave_subscribers.csv"
path2 = r"Session 6_streamwave_sports_events.csv"

# Load the datasets
subscribers = pd.read_csv(path1)
interactions = pd.read_csv(path2)

# Check the data
print('Number of subscribers:', len(subscribers))
print('Number of interactions:', len(interactions))

display(subscribers.head())


Number of subscribers: 1590
Number of interactions: 17


,subscriber_id,monthly_watch_hours,avg_session_minutes,mobile_share,family_profiles,live_sports_hours,kids_content_share,international_content_share,binge_index,price_sensitivity,churn_risk_score,monthly_plan_price,tenure_months,support_tickets_90d
0,SW100000,60.8,38.3,0.51,2,20.7,0.06,0.02,0.32,0.23,0.24,22.82,15.7,2
1,SW100001,24.9,14.9,0.75,1,3.4,0.02,0.04,0.28,0.62,0.06,7.97,6.4,0
2,SW100002,5.4,18.5,0.45,2,2.5,0.00,0.00,0.14,0.68,0.32,5.99,4.6,1
3,SW100003,75.5,32.1,0.17,4,4.0,0.59,0.02,0.63,0.33,0.15,18.79,24.3,1
4,SW100004,20.0,23.2,0.74,1,2.7,0.10,0.01,0.38,0.58,0.26,11.14,12.0,0


In [5]:
cluster_features = [
    'monthly_watch_hours', 'avg_session_minutes', 'mobile_share',
    'family_profiles', 'live_sports_hours', 'kids_content_share',
    'international_content_share', 'binge_index', 'price_sensitivity',
    'churn_risk_score', 'monthly_plan_price', 'tenure_months',
    'support_tickets_90d'
]

X_imputed = SimpleImputer(strategy="median").fit_transform(subscribers[cluster_features])
X_scaled = StandardScaler().fit_transform(X_imputed)
raw_labels = KMeans(n_clusters=4, random_state=42, n_init=25).fit_predict(X_scaled)

# Numeric K-Means labels are arbitrary. Align them with the *behavioral*
# names on the Session 5 slides, even if libraries permute raw label IDs.
raw_profiles = (subscribers.assign(raw_cluster=raw_labels)
                .groupby('raw_cluster')[['live_sports_hours','kids_content_share',
                                         'monthly_watch_hours','mobile_share']].mean())
raw_to_session5_id = {
    int(raw_profiles['live_sports_hours'].idxmax()): 0,   # Sports-focused
    int(raw_profiles['kids_content_share'].idxmax()): 1, # Household viewers
    int(raw_profiles['monthly_watch_hours'].idxmax()): 2, # Heavy viewers
    int(raw_profiles['mobile_share'].idxmax()): 3        # Light mobile users with high churn risk
}
assert len(raw_to_session5_id) == 4, 'Unexpected profiles: inspect the clusters.'
subscribers["cluster"] = pd.Series(raw_labels, index=subscribers.index).map(raw_to_session5_id)

# sizes = subscribers.cluster.value_counts().sort_index()
# display(sizes.rename("number_of_subscribers").to_frame())
# assert sizes.to_dict() == {0: 260, 1: 362, 2: 298, 3: 670}
# print("C0 is our sports-focused cluster (260 accounts).")

## Part B — Link C0 members to *new synthetic* sport-event data

`subscriber_id` is the link between **the original clustering table** and **the additional interactions table**. 

In [6]:
demo = interactions.merge(
    subscribers[['subscriber_id', 'cluster']],
    on='subscriber_id', how='left', validate='many_to_one'
)
assert demo['cluster'].eq(0).all(), "The synthetic demo IDs must be in C0."
display(demo[['teaching_alias','cluster','event_name','interaction_type']])

,teaching_alias,cluster,event_name,interaction_type
0,A,0,Soccer recaps,watched_historical
1,A,0,Champions League previews,watched_historical
2,B,0,Soccer recaps,watched_historical
3,B,0,Champions League previews,watched_historical
4,B,0,Champions League fixture,interested_in_upcoming
5,C,0,Soccer recaps,watched_historical
6,C,0,Champions League previews,watched_historical
7,C,0,Champions League fixture,interested_in_upcoming
8,D,0,NBA recaps,watched_historical
9,D,0,Basketball highlights,watched_historical


## Part C — Baseline: what is most popular among C0 peers?

Subscriber A has **not** expressed an interest in either upcoming event. Count how many other C0 peers have positively indicated an interest in each event. 

In [7]:
target_alias = 'A'
upcoming = demo[demo.interaction_type == 'interested_in_upcoming'].copy()

baseline_scores = (upcoming[upcoming.teaching_alias != target_alias]
                   .groupby('event_name')['teaching_alias']
                   .nunique().sort_values(ascending=False)
                   .rename('interested_C0_peers'))
print('Cluster-popularity baseline:')
display(baseline_scores.to_frame())

Cluster-popularity baseline:


,interested_C0_peers
event_name,
NBA game,3
Champions League fixture,2


## Part D — Collaborative filtering: which C0 peers resemble A?

Build a **binary matrix** from *historical* viewed content, with users in rows and content in columns. Cosine similarity measures how closely those histories align. This simple demo uses zeros for *unobserved* interactions, without claiming they are dislikes.

In [8]:
historical = demo[demo.interaction_type == 'watched_historical']
R = pd.crosstab(historical.teaching_alias, historical.event_name)
R = R.reindex(index=list('ABCDEF'), fill_value=0)
print('Historical viewed-content matrix (0 = unobserved, not disliked):')
display(R)

sim = cosine_similarity(R.loc[[target_alias]], R.loc[list('BCDEF')])[0]
peer_similarity = pd.Series(sim, index=list('BCDEF'), name='cosine_similarity_to_A')
print('Similarity of each peer to Subscriber A:')
display(peer_similarity.to_frame().round(3))

Historical viewed-content matrix (0 = unobserved, not disliked):


event_name,Basketball highlights,Champions League previews,NBA recaps,Soccer recaps
teaching_alias,,,,
A,0,1,0,1
B,0,1,0,1
C,0,1,0,1
D,1,0,1,0
E,1,0,1,0
F,0,0,1,1


Similarity of each peer to Subscriber A:


,cosine_similarity_to_A
B,1.0
C,1.0
D,0.0
E,0.0
F,0.5


## Part E — Score the *upcoming* events using similarity-weighted peers

A simple score for an event is the **sum of similarity weights of peers who expressed positive interest in it**. This is a ranking score, **not a probability**. We can now see why popularity and personalized rankings can disagree.

In [9]:
scored = upcoming.loc[upcoming.teaching_alias != target_alias, ['teaching_alias','event_name']].copy()
scored['similarity_weight'] = scored['teaching_alias'].map(peer_similarity)
weighted = (scored.groupby('event_name')['similarity_weight']
            .sum().sort_values(ascending=False)
            .rename('similarity_weighted_score'))

comparison = pd.concat([baseline_scores, weighted], axis=1).fillna(0)
print('Compare the TWO ranking approaches:')
display(comparison.round(3))
print('Popularity winner:', baseline_scores.index[0])
print('Collaborative-filtering winner:', weighted.index[0])

Compare the TWO ranking approaches:


,interested_C0_peers,similarity_weighted_score
event_name,,
NBA game,3,0.5
Champions League fixture,2,2.0


Popularity winner: NBA game
Collaborative-filtering winner: Champions League fixture


## Part F — Try the recommendation



In [ ]:
top_n = 2

recommendations = weighted.head(top_n).reset_index()
print(f'Top {top_n} personalized sports reminder(s) for subscriber A:')
display(recommendations)


Top 4 personalized sports reminder(s) for subscriber A:


,event_name,similarity_weighted_score
0,Champions League fixture,2.0
1,NBA game,0.5
